# Knowledge Graph Data Processing: Load, Diagnose, Deduplicate, and Export

This notebook is designed to load, diagnose, deduplicate, and export node and relationship data, likely for a knowledge graph.

## Loading Data and Initial Diagnostic Checks




In [ ]:
import pandas as pd

# Define filenames - Ensure these are actual .csv files exported from Google Sheets
nodes_file_name = "/content/dft_kg_nodes_master.csv"
relationships_file_name = "/content/dft_kg_relationships_master.csv"

# Load both raw files (L_raw)
df_nodes_raw = pd.read_csv(nodes_file_name)
df_relationships_raw = pd.read_csv(relationships_file_name)

# --- DIAGNOSTIC FOR RELATIONSHIPS FILE ---

total_relationships = len(df_relationships_raw)
missing_relationships = df_relationships_raw['relationship_type'].isnull().sum()
duplicate_relationships = df_relationships_raw.duplicated().sum()
unique_relationship_types = df_relationships_raw['relationship_type'].nunique()
top_relationships = df_relationships_raw['relationship_type'].value_counts().head(5)


# --- DIAGNOSTIC FOR NODES FILE ---

total_nodes = len(df_nodes_raw)
# Check for null values in the 'label' column (node type classification failure)
missing_node_labels = df_nodes_raw['label'].isnull().sum()
# Check for duplicated node_id entries
duplicate_nodes = df_nodes_raw['node_id'].duplicated().sum()
unique_node_labels = df_nodes_raw['label'].nunique()
top_node_labels = df_nodes_raw['label'].value_counts().head(5)


# --- PRINT RESULTS ---

print("--- DIAGNOSTIC METRICS FOR RAW RELATIONSHIPS (L_raw) ---")
print(f"Total relationships extracted: {total_relationships}")
print(f"Missing Relationship Types (Null Predicates): {missing_relationships}")
print(f"Fully Duplicated Triples (Noise): {duplicate_relationships}")
print(f"Unique Relationship Types Extracted: {unique_relationship_types}")
print("\nTop 5 Most Frequent Relationship Types:")
print(top_relationships.to_string())

print("\n\n--- DIAGNOSTIC METRICS FOR RAW NODES (L_raw) ---")
print(f"Total nodes extracted: {total_nodes}")
print(f"Missing Node Labels (Type Classification Failure): {missing_node_labels}")
print(f"Duplicated Nodes (by node_id): {duplicate_nodes}")
print(f"Unique Node Labels Extracted: {unique_node_labels}")
print("\nTop 5 Most Frequent Node Labels:")
print(top_node_labels.to_string())

--- DIAGNOSTIC METRICS FOR RAW RELATIONSHIPS (L_raw) ---
Total relationships extracted: 5737
Missing Relationship Types (Null Predicates): 1
Fully Duplicated Triples (Noise): 541
Unique Relationship Types Extracted: 11

Top 5 Most Frequent Relationship Types:
relationship_type
RESULT_FOR_BENCHMARK    1342
RESULT_USING_METRIC     1337
RESULT_FOR_METHOD       1337
REPORTS_RESULT           425
APPLIES_CORRECTION       374


--- DIAGNOSTIC METRICS FOR RAW NODES (L_raw) ---
Total nodes extracted: 3263
Missing Node Labels (Type Classification Failure): 0
Duplicated Nodes (by node_id): 1037
Unique Node Labels Extracted: 6

Top 5 Most Frequent Node Labels:
label
ValidationResult        1335
Functional               864
BenchmarkSet             794
Metric                   161
DispersionCorrection      84


## Deduplication of Relationship and Node Data






Automatic deduplication was applied to both the `df_relationships_raw` and `df_nodes_raw` DataFrames to remove redundant entries and establish clean baselines for further analysis. The primary method used for deduplication was the `drop_duplicates()` function from the pandas library.

1.  **Deduplicating Relationships (Triples)**:
    *   **Method**: The `df_relationships_raw` DataFrame was deduplicated by calling `df_relationships_raw.drop_duplicates()` without any arguments. This removes rows that are exact duplicates across all columns, ensuring that each unique triple (subject, predicate, object) is represented only once.
    *   **Impact**: Out of an original count of 5737 relationships, **541 duplicate relationships were removed**. This resulted in a **new baseline count of 5196 unique relationships**.

2.  **Deduplicating Nodes (Entities)**:
    *   **Method**: The `df_nodes_raw` DataFrame was deduplicated using `df_nodes_raw.drop_duplicates(subset=['node_id'], keep='first')`. This specifically targeted duplicates based on the 'node_id' column. By setting `keep='first'`, the first occurrence of each unique `node_id` was retained, establishing a canonical entry for each entity.
    *   **Impact**: From an original count of 3263 nodes, **1037 duplicate nodes were removed** based on their `node_id`. This established a **new baseline count of 2226 unique nodes**.

**Verification**: After deduplication, a check confirmed that 0 duplicates remained in the 'node_id' column of the `df_nodes_deduped` DataFrame, ensuring the success of the node deduplication. The number of unique relationship types in `df_relationships_deduped` remained at 11, indicating that deduplication only removed redundant instances, not unique types.

In [ ]:


# --- AUTOMATIC DEDUPLICATION ---

# 1. Deduplicate Relationships (Removing 634 exact triple duplicates)
df_relationships_deduped = df_relationships_raw.drop_duplicates()
original_rel_count = len(df_relationships_raw)
deduped_rel_count = len(df_relationships_deduped)
rel_duplicates_removed = original_rel_count - deduped_rel_count

# 2. Deduplicate Nodes (Removing 1037 duplicated node_id entries)
# We deduplicate based only on the node_id to keep one canonical entry per unique ID.
df_nodes_deduped = df_nodes_raw.drop_duplicates(subset=['node_id'], keep='first')
original_node_count = len(df_nodes_raw)
deduped_node_count = len(df_nodes_deduped)
node_duplicates_removed = original_node_count - deduped_node_count


# --- ANALYSIS AFTER DEDUPLICATION ---

print("--- AUTOMATIC DEDUPLICATION RESULTS ---")

print("\n[A] RELATIONSHIPS (Triples)")
print(f"Original Count (L_raw): {original_rel_count}")
print(f"Duplicates Removed: {rel_duplicates_removed}")
print(f"**NEW BASELINE TRIPLE COUNT:** {deduped_rel_count}")
print(f"**Remaining Triples for Hallucination Check:** {deduped_rel_count}")

print("\n[B] NODES (Entities)")
print(f"Original Count (L_raw): {original_node_count}")
print(f"Duplicates Removed: {node_duplicates_removed}")
print(f"**NEW BASELINE NODE COUNT:** {deduped_node_count}")
print(f"**Remaining Nodes for Canonicalization Check:** {deduped_node_count}")

# Check for new duplicates in the node file (should be zero if done correctly)
new_node_duplicates = df_nodes_deduped['node_id'].duplicated().sum()
print(f"Nodes check: {new_node_duplicates} duplicates remaining (should be 0)")

# Check for unique relationship types again (should be the same)
print(f"\nUnique Relationship Types: {df_relationships_deduped['relationship_type'].nunique()}")

--- AUTOMATIC DEDUPLICATION RESULTS ---

[A] RELATIONSHIPS (Triples)
Original Count (L_raw): 5737
Duplicates Removed: 541
**NEW BASELINE TRIPLE COUNT:** 5196
**Remaining Triples for Hallucination Check:** 5196

[B] NODES (Entities)
Original Count (L_raw): 3263
Duplicates Removed: 1037
**NEW BASELINE NODE COUNT:** 2226
**Remaining Nodes for Canonicalization Check:** 2226
Nodes check: 0 duplicates remaining (should be 0)

Unique Relationship Types: 11


## Saving and Downloading Deduplicated Data




In [ ]:
df_nodes_deduped.to_csv('nodes_deduped.csv', index=False)
df_relationships_deduped.to_csv('relationships_deduped.csv', index=False)



1.  **Saving Deduplicated Data to CSV**: After the deduplication process, the cleaned node and relationship data are stored in pandas DataFrames named `df_nodes_deduped` and `df_relationships_deduped`. The `.to_csv()` method is used to persist these DataFrames as CSV files (`nodes_deduped.csv` and `relationships_deduped.csv`). The `index=False` argument is crucial here, as it prevents pandas from writing the DataFrame index as a column in the CSV, which is typically undesirable for clean data export.

2.  **Making Files Available for Download**: The `google.colab.files.download()` function is then called for each of the newly created CSV files. This function initiates a download prompt in the Colab environment, allowing the user to easily retrieve the cleaned `nodes_deduped.csv` and `relationships_deduped.csv` files to their local machine. This is a convenient way to export processed data from the Colab runtime for further use or analysis outside of the notebook.

In [ ]:
from google.colab import files

files.download('nodes_deduped.csv')
files.download('relationships_deduped.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>